In [0]:
import dlt
from pyspark.sql import functions as F

In [0]:
#  Expectations
rules = {
    'product_id' : 'product_id IS NOT NULL',
    'product_name' : 'product_name IS NOT NULL'
}

In [0]:
@dlt.table(name="products_view")
@dlt.expect_all_or_drop(rules)
def bronze_products():
    df = spark.readStream.table("project.bronze.products")
    return product_trans(df)

In [0]:
def product_trans(df):
    df = df.drop("_rescued_data")
    df = df.withColumn('discounted_price', F.col('price')*0.90)
    df = df.withColumn("event_ts", F.current_timestamp())
    return df


In [0]:
dlt.create_streaming_table(name="project.silver.products_scd1")

In [0]:
# @dlt.table
# def product_for_scd2():
#     df = spark.read.table("products_view")
#     df = product_trans(df)
#     return df

In [0]:
dlt.apply_changes(
    target = 'project.silver.products_scd1',
    source = 'products_view',
    keys= ['product_id'],
    sequence_by = 'event_ts',
    stored_as_scd_type = 1,
    name = "apply_changes_products_silver"
)



In [0]:
# df = spark.read.format('parquet') \
#     .load('/Volumes/project/bronze/raw_products/data')

# display(df)

In [0]:
# df = df.drop("_rescued_data")

In [0]:
# df.createOrReplaceTempView("products")

In [0]:
# %sql
# CREATE OR REPLACE FUNCTION project.bronze.discount_func(p double)
# RETURNS double
# RETURN p * 0.90

In [0]:
# %sql
# SELECT product_id, product_name, price, round(project.bronze.discount_func(price),2) as discounted_price FROM products limit 10;


In [0]:
# from pyspark.sql import functions as F


In [0]:
# %sql
# CREATE OR REPLACE FUNCTION project.bronze.upper_func(st STRING)
# RETURNS STRING
# LANGUAGE PYTHON
# AS 
# $$
#     return st.upper() if st is not None else None
# $$




In [0]:
# df.selectExpr("product_id", "product_name", "price", "project.bronze.upper_func(brand) AS brand_u").limit(10).display()

In [0]:
# df.write.format("delta").mode("overwrite").save("/Volumes/project/volumes/silver_products_volume/data")

In [0]:
# df.write.format('delta').mode('append').saveAsTable('project.silver.products')

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS project.silver.products
# AS 
# SELECT * FROM DELTA.`/Volumes/project/volumes/silver_products_volume/data`